In [4]:
import requests


In [5]:
repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

In [6]:
zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'

print(zip_url)

https://github.com/evidentlyai/docs/archive/refs/heads/main.zip


In [7]:
zip_response = requests.get(zip_url)

In [8]:
zip_response.status_code

200

In [9]:
import io
import zipfile

In [10]:
zip_archive = zipfile.ZipFile(
    io.BytesIO(zip_response.content)
)

In [11]:
zip_archive.namelist()[:10]

['docs-main/',
 'docs-main/api-reference/',
 'docs-main/api-reference/endpoint/',
 'docs-main/api-reference/endpoint/create.mdx',
 'docs-main/api-reference/endpoint/delete.mdx',
 'docs-main/api-reference/endpoint/get.mdx',
 'docs-main/api-reference/introduction.mdx',
 'docs-main/api-reference/openapi.json',
 'docs-main/changelog/',
 'docs-main/changelog/changelog.mdx']

In [12]:
mdx_file = zip_archive.open(
    'docs-main/api-reference/endpoint/create.mdx'
)

In [13]:
mdx_content = mdx_file.read().decode('utf-8')

In [14]:
print(mdx_content[:500])

---
title: 'Create Plant'
openapi: 'POST /plants'
---



In [15]:
import frontmatter

In [16]:
post = frontmatter.loads(mdx_content)

In [17]:
post.metadata

{'title': 'Create Plant', 'openapi': 'POST /plants'}

In [18]:
post.content

''

In [19]:
filenames = zip_archive.namelist()

In [20]:
len(filenames)

278

In [21]:
for filename in filenames:
    if filename.endswith(('.md', '.mdx')):
        print(filename)

docs-main/api-reference/endpoint/create.mdx
docs-main/api-reference/endpoint/delete.mdx
docs-main/api-reference/endpoint/get.mdx
docs-main/api-reference/introduction.mdx
docs-main/changelog/changelog.mdx
docs-main/docs/library/data_definition.mdx
docs-main/docs/library/descriptors.mdx
docs-main/docs/library/evaluations_overview.mdx
docs-main/docs/library/leftover_content.mdx
docs-main/docs/library/metric_generator.mdx
docs-main/docs/library/output_formats.mdx
docs-main/docs/library/overview.mdx
docs-main/docs/library/prompt_optimization.mdx
docs-main/docs/library/report.mdx
docs-main/docs/library/synthetic_data_api.mdx
docs-main/docs/library/tags_metadata.mdx
docs-main/docs/library/tests.mdx
docs-main/docs/platform/alerts.mdx
docs-main/docs/platform/dashboard_add_panels.mdx
docs-main/docs/platform/dashboard_add_panels_ui.mdx
docs-main/docs/platform/dashboard_overview.mdx
docs-main/docs/platform/dashboard_panel_types.mdx
docs-main/docs/platform/datasets_generate.mdx
docs-main/docs/platf

In [22]:
documents = []

for filename in filenames:
    if filename.endswith(('.md', '.mdx')):
        file = zip_archive.open(filename)
        content = file.read().decode('utf-8')

        post = frontmatter.loads(content)

        document = {
            "filename": filename,
            "metadata": post.metadata,
            "content": post.content
        }
        documents.append(document)
        

In [23]:
len(documents)

95

In [24]:
documents[0][:500]

KeyError: slice(None, 500, None)

In [ ]:
documents[0]

{'filename': 'docs-main/api-reference/endpoint/create.mdx',
 'metadata': {'title': 'Create Plant', 'openapi': 'POST /plants'},
 'content': ''}

In [ ]:
for document in documents:
    if document["content"]:
        print(document["filename"])
        print(document["content"][:500])
        break

docs-main/api-reference/introduction.mdx
<Note>
  If you're not looking to build API reference documentation, you can delete
  this section by removing the api-reference folder.
</Note>

## Welcome

There are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.

<Card
  title="Plant Store Endpoints"
  icon="leaf"
  href="https://github.


In [ ]:
documents[1]

{'filename': 'docs-main/api-reference/endpoint/delete.mdx',
 'metadata': {'title': 'Delete Plant', 'openapi': 'DELETE /plants/{id}'},
 'content': ''}

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [ ]:
files[0]

RawRepositoryFile(filename='api-reference/endpoint/create.mdx', content="---\ntitle: 'Create Plant'\nopenapi: 'POST /plants'\n---")

In [ ]:
documents = []

for file in files:
    document = file.parse()
    documents.append(document)

In [ ]:
len(documents)

95

In [ ]:
from minsearch import Index

In [ ]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)

In [25]:
index.fit(documents)

In [26]:
query = "LLM as a Judge"

In [28]:
results = index.search(query, num_results=5)


In [29]:
len(results)

5

In [30]:
results[0]

{'filename': 'docs-main/docs/library/prompt_optimization.mdx',
 'metadata': {'title': 'Prompt optimization',
  'description': '[NEW] Automated prompt optimization.'},
 'content': 'More detailed documentation coming soon. \n\nRead the release blog on [prompt optimization for LLM judges](https://www.evidentlyai.com/blog/llm-judge-prompt-optimization).\n\nExample notebooks:\n- Code review binary LLM judge prompt optimization: [code example](https://github.com/evidentlyai/evidently/blob/main/examples/cookbook/prompt_optimization_code_review_example.ipynb)\n- Topic multi-class LLM judge prompt optimization: [code example](https://github.com/evidentlyai/evidently/blob/main/examples/cookbook/prompt_optimization_bookings_example.ipynb)\n- Tweet generation prompt optimization: [code example](https://github.com/evidentlyai/evidently/blob/main/examples/cookbook/prompt_optimization_tweet_generation_example.ipynb)'}

In [31]:
results[1]

{'filename': 'docs-main/examples/introduction.mdx',
 'metadata': {'title': 'Tutorials and guides',
  'description': 'End-to-end code examples.'},
 'content': '<Note>\n  **We have an applied course on LLM evaluations\\!** Free video course with 10\\+ tutorials. [Sign up](https://www.evidentlyai.com/llm-evaluation-course-practice).\n</Note>\n\n## Quickstarts\n\nIf you are new, start here.\n\n<CardGroup cols={3}>\n  <Card title="LLM quickstart" icon="comment-text" href="/quickstart_llm">\n    Evaluate the quality of text outputs.\n  </Card>\n  <Card title="ML quickstart" icon="table" href="/quickstart_ml">\n    Test tabular data quality and data drift.\n  </Card>\n  <Card title="Tracing quickstart" icon="bars-staggered" href="/quickstart_tracing">\n    Collect inputs and outputs from AI your app.\n  </Card>\n</CardGroup>\n\n## LLM Tutorials\n\nEnd-to-end examples of specific workflows and use cases.\n\n<CardGroup cols={2}>\n  <Card title="LLM as a judge" icon="scale-balanced" href="/examp